In [1]:
import re
import pandas as pd

def normalize_skill(raw_text, skill_lookup):
    text = str(raw_text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # remove special characters (-, ., etc.)
    text = re.sub(r"\s+", " ", text).strip()    # collapse repeated whitespace

    # Exact alias/canonical match first
    if text in skill_lookup:
        return skill_lookup[text]

    # Try removing common trailing noise words like "programming", "framework"
    stripped = re.sub(r"\b(programming|framework|language)\b", "", text).strip()
    if stripped in skill_lookup:
        return skill_lookup[stripped]

    return None  # not a recognized skill

In [3]:
import os
import sys

# Add the skill_extractor folder to Python's search path
BASE_DIR = os.getcwd()
sys.path.append(os.path.join(BASE_DIR, "..", "skill_extractor"))

from rule_based_extractor import load_skill_list

skill_lookup = load_skill_list()

In [4]:
df = pd.read_csv("../data/clean_jobs.csv")

# If you have saved outputs from various methods, load and normalize each
rule_based = pd.read_csv("../data/rule_based_skills.csv")
regex_df = pd.read_csv("../data/eval_sample.csv")  # or wherever regex_skills live

def normalize_skill_list(skill_list, skill_lookup):
    if isinstance(skill_list, str):
        import ast
        skill_list = ast.literal_eval(skill_list)  # CSV stores lists as strings
    normalized = set()
    for s in skill_list:
        canon = normalize_skill(s, skill_lookup)
        if canon:
            normalized.add(canon)
    return sorted(normalized)

rule_based["normalized_skills"] = rule_based["rule_based_skills"].apply(
    lambda x: normalize_skill_list(x, skill_lookup)
)

In [5]:
output = rule_based[["job_id", "job_title", "normalized_skills"]]
output.to_csv("../data/normalized_skills.csv", index=False)
print(f"Saved {len(output)} normalized rows")

Saved 10000 normalized rows


In [6]:
# test Cases

test_cases = {
    "python3": "Python",
    "python 3": "Python",
    "Python programming": "Python",
    "powerbi": "Power BI",
    "power-bi": "Power BI",
    "postgres": "PostgreSQL",
}

print(f"{'Input':<25} {'Expected':<15} {'Got':<15} {'Match?'}")
print("-" * 65)
all_passed = True
for raw, expected in test_cases.items():
    got = normalize_skill(raw, skill_lookup)
    match = "✅" if got == expected else "❌"
    if got != expected:
        all_passed = False
    print(f"{raw:<25} {expected:<15} {str(got):<15} {match}")

print("\nAll tests passed!" if all_passed else "\nSome tests FAILED — check output above.")

Input                     Expected        Got             Match?
-----------------------------------------------------------------
python3                   Python          Python          ✅
python 3                  Python          Python          ✅
Python programming        Python          Python          ✅
powerbi                   Power BI        Power BI        ✅
power-bi                  Power BI        Power BI        ✅
postgres                  PostgreSQL      PostgreSQL      ✅

All tests passed!


In [7]:
print("postgres" in skill_lookup)
print("powerbi" in skill_lookup)
print("power bi" in skill_lookup)  # note: space, not hyphen — this is the form that actually matters

True
True
True


In [8]:
import ast

sample_terms = []
for skills in rule_based["rule_based_skills"].head(50):
    if isinstance(skills, str):
        skills = ast.literal_eval(skills)
    sample_terms.extend(skills)

results = [(t, normalize_skill(t, skill_lookup)) for t in sample_terms]
unmatched = [t for t, canon in results if canon is None]

print(f"Total terms checked: {len(results)}")
print(f"Unmatched (returned None): {len(unmatched)}")
print(f"Unmatched examples: {unmatched[:15]}")

Total terms checked: 259
Unmatched (returned None): 0
Unmatched examples: []


# Before / After Comparison

In [9]:
raw_skill_set = set()
for skills in rule_based["rule_based_skills"]:
    if isinstance(skills, str):
        skills = ast.literal_eval(skills)
    raw_skill_set.update(skills)

normalized_skill_set = set()
for skills in rule_based["normalized_skills"]:
    normalized_skill_set.update(skills)

print(f"Distinct raw skill strings before normalization: {len(raw_skill_set)}")
print(f"Distinct canonical skills after normalization:   {len(normalized_skill_set)}")

Distinct raw skill strings before normalization: 46
Distinct canonical skills after normalization:   46
